# RoadSense: Robust Traffic Sign Recognition
## Phase 4: Robustness Improvement & Transfer Learning
**Group 4 | Atria University**  
- **Team Leader:** Aseem  
- **Primary Owner (Phase 4):** Manoj  
- **Members:** Raju, Sohail  

---
### Objectives of Phase 4:
1. **Phase 4A — Improved CNN:** Train the same scratch CNN architecture with label-safe augmentations (rotation, affine, colour jitter) and re-run the full robustness benchmark to show measurable improvement over the baseline.
2. **Phase 4B — Transfer Learning:** Fine-tune two pretrained ImageNet backbones — **ResNet-18** and **MobileNetV2** — on the 43-class GTSRB dataset using a two-stage strategy (frozen backbone → full fine-tune).
3. **Three-Way Comparison:** Produce a single fair comparison table of Baseline CNN vs Improved CNN vs ResNet-18 vs MobileNetV2 on both clean test set and robustness benchmark.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.models import BaselineCNN, get_transfer_model, count_parameters, count_all_parameters
from src.dataset import get_data_loaders, DATA_DIR
from src.train import train
from src.evaluate import evaluate
from src.robustness_bench import run_robustness_benchmark
from src.utils import CLASS_NAMES, set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style='whitegrid')
FIG_DIR = ROOT_DIR / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Active device : {device}')
print(f'Root directory: {ROOT_DIR}')

---
## Part A — Improved CNN with Label-Safe Augmentations

### Why Augmentation?
The robustness test bench (Phase 3) revealed the biggest accuracy drops under:
- **Brightness/Contrast** degradation (night driving, glare)
- **Gaussian Noise** (noisy camera sensors)
- **Motion Blur** (vehicle movement, defocus)

We train the **identical BaselineCNN architecture** but now expose it to these transformations during training, so the model learns features that are invariant to them.

### Augmentation Strategy (Label-Safe Only):
| Augmentation | Parameters | Why Label-Safe |
| :--- | :--- | :--- |
| **RandomRotation** | ±10° | Simulates camera tilt; signs remain same class |
| **RandomAffine** | translate=(6%, 6%), scale=(0.95–1.05) | Simulates position variation; no semantic change |
| **ColorJitter** | brightness=0.2, contrast=0.2, saturation=0.2 | Simulates lighting conditions |
| ❌ Horizontal Flip | **NEVER** | Would convert Keep-Left (39) to Keep-Right (38) |
| ❌ Vertical Flip | **NEVER** | Inverts directional and Yield signs |

In [ ]:
# ============================================================
# Train Improved CNN (augmented) — Phase 4A
# ============================================================
# This uses the SAME BaselineCNN architecture as Phase 2,
# but training data is augmented with label-safe transforms.

print('Training Improved CNN with label-safe augmentations...')
print('This may take 10–20 minutes on CPU (3–5 min on GPU)\n')

improved_model, improved_history = train(
    epochs=15,
    batch_size=64,
    lr=0.001,
    use_augmentation=True,       # KEY DIFFERENCE from Phase 2
    use_class_weights=True,
    model_type='improved_cnn',
    model_name='improved_cnn',
    stage1_epochs=5,             # Not used for scratch CNN but required arg
)

In [ ]:
# ============================================================
# Evaluate Improved CNN on Clean Test Set
# ============================================================
improved_metrics, improved_confused = evaluate(model_name='improved_cnn')

In [ ]:
# ============================================================
# Run Robustness Benchmark for Improved CNN
# ============================================================
print('\nRunning robustness benchmark on improved_cnn...')
improved_robustness, improved_clean_acc, improved_mca = run_robustness_benchmark(
    model_name='improved_cnn'
)

In [ ]:
# ============================================================
# Phase 4A: Before vs After Augmentation Comparison
# ============================================================

baseline_rob_path  = ROOT_DIR / 'results' / 'baseline_cnn_robustness.csv'
improved_rob_path  = ROOT_DIR / 'results' / 'improved_cnn_robustness.csv'

df_baseline_rob = pd.read_csv(baseline_rob_path)
df_improved_rob = pd.read_csv(improved_rob_path)

# Mean per corruption type
baseline_by_corr = df_baseline_rob.groupby('Corruption')['Accuracy_%'].mean().reset_index()
improved_by_corr = df_improved_rob.groupby('Corruption')['Accuracy_%'].mean().reset_index()

merged = baseline_by_corr.merge(improved_by_corr, on='Corruption', suffixes=('_baseline', '_improved'))
merged['Gain_%'] = merged['Accuracy_%_improved'] - merged['Accuracy_%_baseline']
merged['Corruption'] = merged['Corruption'].str.replace('_', ' ').str.title()

print('\nPhase 4A — Robustness Comparison (avg over 3 severities):')
print(merged.to_string(index=False))

# Bar chart comparison
x = np.arange(len(merged))
w = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - w/2, merged['Accuracy_%_baseline'], w, label='Baseline CNN (no aug)', color='#2b5c8f', alpha=0.85)
bars2 = ax.bar(x + w/2, merged['Accuracy_%_improved'],  w, label='Improved CNN (+aug)', color='#81b29a', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(merged['Corruption'], rotation=15, ha='right')
ax.set_ylabel('Mean Accuracy (%) across 3 severity levels')
ax.set_title('Phase 4A: Robustness Improvement through Label-Safe Augmentation')
ax.legend()
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

# Annotate gain
for i, gain in enumerate(merged['Gain_%']):
    colour = '#2d6a4f' if gain >= 0 else '#c0392b'
    ax.text(x[i] + w/2, merged['Accuracy_%_improved'].iloc[i] + 1.0,
            f'+{gain:.1f}%' if gain >= 0 else f'{gain:.1f}%',
            ha='center', va='bottom', fontsize=9, color=colour, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'phase4a_augmentation_comparison.png', dpi=300)
plt.show()
print(f"Saved to {FIG_DIR / 'phase4a_augmentation_comparison.png'}")

---
## Part B — Transfer Learning: ResNet-18 & MobileNetV2

### Architecture Rationale

| Architecture | Params (total) | Trainable (Stage 1) | Design Focus |
| :--- | :---: | :---: | :--- |
| **ResNet-18** | ~11.7M | ~550K (head only) | Deep residual learning; excellent spatial feature extraction |
| **MobileNetV2** | ~3.4M | ~340K (head only) | Inverted residuals + depthwise separable convolutions; optimized for mobile/edge |

### Two-Stage Fine-Tuning Strategy

```
Stage 1 (epochs 1–5)  : Backbone FROZEN (pretrained ImageNet weights preserved)
                         Only the new 43-class classification head is trained.
                         LR = 1e-3 (head only)
                         Purpose: Warm up the head without corrupting ImageNet features
                         with a large gradient signal on small-domain GTSRB images.

Stage 2 (epochs 6–15) : All layers UNFROZEN → full end-to-end fine-tuning
                         LR backbone = 1e-4 (10x lower than head)
                         LR head     = 1e-3
                         Purpose: Gently adapt pretrained features to GTSRB domain
                         (small 48×48 signs) without catastrophic forgetting.
```

> **Input size:** Transfer models use **48×48** instead of 32×32 to better utilize the pretrained backbone's spatial resolution expectations.

In [ ]:
# ============================================================
# Inspect Transfer Model Architectures
# ============================================================

print('=== ResNet-18 (frozen backbone) ===')
r18 = get_transfer_model('resnet18', num_classes=43, freeze_backbone=True)
r18_trainable = count_parameters(r18)
r18_total     = count_all_parameters(r18)
print(f'  Trainable params (Stage 1) : {r18_trainable:,}')
print(f'  Total params               : {r18_total:,}')

print('\n=== MobileNetV2 (frozen backbone) ===')
mv2 = get_transfer_model('mobilenet_v2', num_classes=43, freeze_backbone=True)
mv2_trainable = count_parameters(mv2)
mv2_total     = count_all_parameters(mv2)
print(f'  Trainable params (Stage 1) : {mv2_trainable:,}')
print(f'  Total params               : {mv2_total:,}')

# Verify forward pass shapes
dummy = torch.randn(4, 3, 48, 48)
with torch.no_grad():
    r18.eval(); out_r18 = r18(dummy)
    mv2.eval(); out_mv2 = mv2(dummy)
print(f'\nResNet-18  output shape: {out_r18.shape}  (expected: [4, 43])')
print(f'MobileNetV2 output shape: {out_mv2.shape}  (expected: [4, 43])')

In [ ]:
# ============================================================
# Train ResNet-18 (Two-Stage Fine-Tuning)
# ============================================================
print('Training ResNet-18...')
print('Stage 1 (epochs 1-5):  Only head trains (backbone frozen)')
print('Stage 2 (epochs 6-15): Full fine-tuning with differential LR\n')

r18_model, r18_history = train(
    epochs=15,
    batch_size=64,
    lr=0.001,
    use_augmentation=True,        # Augmentation applied for robustness
    use_class_weights=True,
    model_type='resnet18',
    model_name='resnet18',
    stage1_epochs=5,
)

In [ ]:
# ============================================================
# Evaluate ResNet-18 on Clean Test Set
# ============================================================
r18_metrics, r18_confused = evaluate(model_name='resnet18')

In [ ]:
# ============================================================
# Robustness Benchmark for ResNet-18
# ============================================================
print('\nRunning robustness benchmark on ResNet-18...')
r18_robustness, r18_clean_acc, r18_mca = run_robustness_benchmark(
    model_name='resnet18'
)

In [ ]:
# ============================================================
# Train MobileNetV2 (Two-Stage Fine-Tuning)
# ============================================================
print('Training MobileNetV2...')

mv2_model, mv2_history = train(
    epochs=15,
    batch_size=64,
    lr=0.001,
    use_augmentation=True,
    use_class_weights=True,
    model_type='mobilenet_v2',
    model_name='mobilenet_v2',
    stage1_epochs=5,
)

In [ ]:
# ============================================================
# Evaluate MobileNetV2 on Clean Test Set
# ============================================================
mv2_metrics, mv2_confused = evaluate(model_name='mobilenet_v2')

In [ ]:
# ============================================================
# Robustness Benchmark for MobileNetV2
# ============================================================
print('\nRunning robustness benchmark on MobileNetV2...')
mv2_robustness, mv2_clean_acc, mv2_mca = run_robustness_benchmark(
    model_name='mobilenet_v2'
)

---
## Full Four-Way Model Comparison
### Clean Test Set + Robustness Benchmark

In [ ]:
# ============================================================
# Build Unified Comparison Table
# ============================================================

# Load all metrics from saved CSV (includes Phase 2 baseline)
metrics_csv = ROOT_DIR / 'results' / 'metrics.csv'
df_metrics = pd.read_csv(metrics_csv)

# Load robustness results
def get_mca(model_name):
    p = ROOT_DIR / 'results' / f'{model_name}_robustness.csv'
    if p.exists():
        df = pd.read_csv(p)
        return round(df['Accuracy_%'].mean(), 2)
    return None

model_order = ['baseline_cnn', 'improved_cnn', 'resnet18', 'mobilenet_v2']
display_names = {
    'baseline_cnn' : 'Baseline CNN (scratch, no aug)',
    'improved_cnn' : 'Improved CNN (scratch + aug)',
    'resnet18'     : 'ResNet-18 (fine-tuned)',
    'mobilenet_v2' : 'MobileNetV2 (fine-tuned)',
}
params_map = {
    'baseline_cnn' : '823K',
    'improved_cnn' : '823K',
    'resnet18'     : '11.7M',
    'mobilenet_v2' : '3.4M',
}

rows = []
for m in model_order:
    row_m = df_metrics[df_metrics['Model'] == m]
    if row_m.empty:
        continue
    row_m = row_m.iloc[0]
    rows.append({
        'Model':              display_names.get(m, m),
        'Parameters':         params_map.get(m, '?'),
        'Clean Acc (%)':      row_m['Clean_Accuracy_%'],
        'Macro-F1 (%)':       row_m['Macro_F1_%'],
        'Mean Corr Acc (%)':  get_mca(m),
    })

df_compare = pd.DataFrame(rows)
print('\n=== FOUR-WAY MODEL COMPARISON ===')
print(df_compare.to_string(index=False))

# Save to CSV
df_compare.to_csv(ROOT_DIR / 'results' / 'model_comparison.csv', index=False)
print(f"\nSaved comparison table to results/model_comparison.csv")

In [ ]:
# ============================================================
# Grouped Bar Chart: Clean Accuracy vs Mean Corruption Accuracy
# ============================================================

fig, ax = plt.subplots(figsize=(13, 6))

model_labels = df_compare['Model'].tolist()
clean_accs   = df_compare['Clean Acc (%)'].tolist()
mca_accs     = df_compare['Mean Corr Acc (%)'].tolist()

x = np.arange(len(model_labels))
w = 0.35

bars1 = ax.bar(x - w/2, clean_accs, w, label='Clean Test Set Accuracy', color='#2b5c8f', alpha=0.9)
bars2 = ax.bar(x + w/2, mca_accs,   w, label='Mean Corruption Accuracy (mCA)', color='#e07a5f', alpha=0.9)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=12, ha='right', fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_title('RoadSense: Four-Way Model Comparison\nClean Accuracy vs Mean Corruption Accuracy (mCA)', fontsize=13)
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'four_way_model_comparison.png', dpi=300)
plt.show()
print(f"Saved to {FIG_DIR / 'four_way_model_comparison.png'}")

In [ ]:
# ============================================================
# Overlay Robustness Degradation Curves (All Models)
# ============================================================

model_configs = [
    ('baseline_cnn',  'Baseline CNN',        '#2b5c8f', '-'),
    ('improved_cnn',  'Improved CNN (+aug)',  '#81b29a', '--'),
    ('resnet18',      'ResNet-18',            '#e07a5f', '-.'),
    ('mobilenet_v2',  'MobileNetV2',          '#f4a261', ':'),
]

corruption_names = ['gaussian_noise', 'motion_blur', 'brightness_contrast',
                    'random_occlusion', 'jpeg_compression']

fig, axes = plt.subplots(1, len(corruption_names), figsize=(20, 5), sharey=True)

for ax, corr in zip(axes, corruption_names):
    for model_id, label, color, ls in model_configs:
        p = ROOT_DIR / 'results' / f'{model_id}_robustness.csv'
        if not p.exists():
            continue
        df = pd.read_csv(p)
        df_corr = df[df['Corruption'] == corr].sort_values('Severity')
        ax.plot(
            df_corr['Severity'].values,
            df_corr['Accuracy_%'].values,
            marker='o', linestyle=ls, color=color, label=label, linewidth=2
        )
    ax.set_title(corr.replace('_', '\n').title(), fontsize=9)
    ax.set_xlabel('Severity')
    ax.set_xticks([1, 2, 3])
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Accuracy (%)')

handles = [mpatches.Patch(color=c, label=l) for _, l, c, _ in model_configs]
fig.legend(handles=handles, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Robustness Degradation Curves: All Models vs 5 Corruption Types', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'all_models_robustness_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved to {FIG_DIR / 'all_models_robustness_overlay.png'}")

In [ ]:
# ============================================================
# Learning Curves Comparison (all 4 models in one figure)
# ============================================================

model_hist_configs = [
    ('baseline_cnn',  'Baseline CNN',       '#2b5c8f'),
    ('improved_cnn',  'Improved CNN (+aug)', '#81b29a'),
    ('resnet18',      'ResNet-18',           '#e07a5f'),
    ('mobilenet_v2',  'MobileNetV2',         '#f4a261'),
]

fig, axs = plt.subplots(1, 2, figsize=(16, 6))

for model_id, label, color in model_hist_configs:
    history_path = ROOT_DIR / 'results' / f'{model_id}_history.json'
    if not history_path.exists():
        print(f'  Skipping {model_id} (no history file found)')
        continue
    with open(history_path) as f:
        h = json.load(f)
    epochs = range(1, len(h['val_acc']) + 1)
    axs[0].plot(epochs, h['val_loss'], marker='.', color=color, label=label)
    axs[1].plot(epochs, [a*100 for a in h['val_acc']], marker='.', color=color, label=label)

axs[0].set_title('Validation Loss — All Models')
axs[0].set_xlabel('Epoch'); axs[0].set_ylabel('Loss')
axs[0].legend(); axs[0].grid(True, alpha=0.3)

axs[1].set_title('Validation Accuracy — All Models')
axs[1].set_xlabel('Epoch'); axs[1].set_ylabel('Accuracy (%)')
axs[1].legend(); axs[1].grid(True, alpha=0.3)

plt.suptitle('Phase 4: Learning Curves Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'phase4_learning_curves_comparison.png', dpi=300)
plt.show()
print(f"Saved to {FIG_DIR / 'phase4_learning_curves_comparison.png'}")

---
## Phase 4 — Summary & Discussion

### Key Findings:

**Phase 4A (Improved CNN):**
- Adding label-safe augmentations to the same scratch CNN architecture directly addresses the corruption types the model was most vulnerable to.
- The clean test accuracy may drop slightly (±0.5%) because the model sees harder training examples, but Mean Corruption Accuracy (mCA) improves — demonstrating real robustness gains.
- This validates the principle: *if a model has never seen a corruption during training, it has no reason to be invariant to it.*

**Phase 4B (Transfer Learning):**
- ResNet-18 and MobileNetV2, pretrained on ImageNet, arrive with rich low-level feature detectors (edges, textures, shapes) already learned from 1.2M diverse images.
- Two-stage fine-tuning prevents **catastrophic forgetting**: the backbone's ImageNet features are preserved in Stage 1 and then gently adapted in Stage 2 with a 10× lower LR.
- Transfer models generally achieve higher clean accuracy with fewer GTSRB training samples compared to scratch CNNs.

**Limitations:**
- Transfer models use 48×48 input; the backbone was pretrained on 224×224 images. The domain gap (large ImageNet objects vs small traffic signs) means the pretrained features are not a perfect fit.
- MobileNetV2's depthwise separable convolutions, designed for mobile inference, may produce different robustness characteristics than ResNet-18's full convolutions.

---
### Next Step: Phase 5 — Grad-CAM Explainability
Open `notebooks/04_evaluation_and_explainability.ipynb` to generate Grad-CAM heatmaps for clean and corrupted images and compare where the models are "looking" when they predict correctly vs incorrectly.